### Cleaning the Data

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
OUT = Path("../output")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

raw = pd.read_csv(DATA / "renewals_raw_extract.csv")
df = raw.copy()

with open(DATA / "pricing_rules.json") as f:
    rules = json.load(f)

log = [
    {"step": "raw extract", "rows": len(df), "dropped": 0}
]


def log_step(label):
    dropped = log[-1]["rows"] - len(df)
    log.append({"step": label, "rows": len(df), "dropped": dropped})
    print(f"{label:<34}: {len(df)} rows (-{dropped})")


print(f"{'raw extract':<34}: {len(df)} rows")

raw extract                       : 813 rows


### 1. Standardize Identifiers, Categories, Booleans and Dates

In [3]:
df["subscription_id"] = df["subscription_id"].str.strip().str.upper()

for column in ["region", "segment", "tier", "product"]:
    df[column] = df[column].str.strip()

odd_ids = df.loc[~df["subscription_id"].str.fullmatch(r"SUB-\d{5}"), "subscription_id"]
print("IDs not matching the Pattern ...", odd_ids.tolist())

IDs not matching the Pattern ... []


In [4]:
official = {
    "region": {
        "americas": "Americas", "amer": "Americas",
        "emea": "EMEA",
        "apjc": "APJC", "apac": "APJC",
    },
    "segment": {
        "smb": "SMB", "small business": "SMB",
        "mid-market": "Mid-Market", "mid market": "Mid-Market",
        "enterprise": "Enterprise", "ent": "Enterprise",
        "public sector": "Public Sector",
    },
    "tier": {
        "essentials": "Essentials", "essential": "Essentials",
        "advantage": "Advantage", "adv": "Advantage",
        "premier": "Premier", "prem": "Premier",
    },
    "product": {
        "corvana collaborate": "Corvana Collaborate",
        "corvana secureedge": "Corvana SecureEdge", "secureedge": "Corvana SecureEdge",
        "corvana netops cloud": "Corvana NetOps Cloud", "netops cloud": "Corvana NetOps Cloud",
        "corvana observe": "Corvana Observe",
    },
}

for column, mapping in official.items():
    mapped = df[column].str.lower().map(mapping)
    unmapped = df.loc[mapped.isna(), column].unique().tolist()
    print(f"{column}: unmapped = {unmapped}")
    assert not unmapped, f"{column} has spellings that are missing from the dictionary"
    df[column] = mapped

print()
for column in official:
    print(f"{column}: {df[column].value_counts().to_dict()}")

region: unmapped = []
segment: unmapped = []
tier: unmapped = []
product: unmapped = []

region: {'Americas': 389, 'EMEA': 228, 'APJC': 196}
segment: {'SMB': 296, 'Mid-Market': 237, 'Enterprise': 158, 'Public Sector': 122}
tier: {'Essentials': 360, 'Advantage': 318, 'Premier': 135}
product: {'Corvana Collaborate': 328, 'Corvana SecureEdge': 218, 'Corvana NetOps Cloud': 170, 'Corvana Observe': 97}


In [5]:
auto_renew_map = {
    "Yes": True, "Y": True, "TRUE": True, "1": True,
    "No": False, "N": False, "FALSE": False, "0": False,
}

auto_renew = df["auto_renew"].str.strip().map(auto_renew_map)
assert auto_renew.notna().all(), "auto_renew has spellings that are missing from the dictionary"
df["auto_renew"] = auto_renew.astype(bool)

print(f"auto_renew: {df['auto_renew'].value_counts().to_dict()}")

auto_renew: {False: 495, True: 318}


In [6]:
iso = pd.to_datetime(df["renewal_date"], format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(df["renewal_date"], format="%d/%m/%Y", errors="coerce")

df["renewal_date"] = iso.fillna(dmy)

print("Parsed as YYYY-MM-DD : ", iso.notna().sum())
print("Parsed as DD/MM/YYYY : ", (iso.isna() & dmy.notna()).sum())
print("Still not a date     : ", df["renewal_date"].isna().sum())
print("Range                : ", df["renewal_date"].min().date(), "to", df["renewal_date"].max().date())

log_step("1. standardized")

Parsed as YYYY-MM-DD :  677
Parsed as DD/MM/YYYY :  136
Still not a date     :  0
Range                :  2025-10-01 to 2026-09-30
1. standardized                   : 813 rows (-0)


### 2. Remove Duplicates

In [7]:
df = df.drop_duplicates()
log_step("2a. exact duplicates dropped")

# the later row is the refreshed one, so keep the last
df = df.drop_duplicates(subset="subscription_id", keep="last").reset_index(drop=True)
log_step("2b. duplicate ids dropped")

print("Duplicated Subscription Ids :", df["subscription_id"].duplicated().sum())

2a. exact duplicates dropped      : 801 rows (-12)
2b. duplicate ids dropped         : 795 rows (-6)
Duplicated Subscription Ids : 0


### 3. Fill Telemetry Gaps

The usage platform had outages. Utilization needs a complete monthly series first.

#### A. Load the Raw Monthly Usage

In [8]:
usage = (
    pd.read_csv(DATA / "usage_raw_monthly.csv")
    .sort_values(["subscription_id", "month"])
    .reset_index(drop=True)
)
usage["missing"] = usage["active_users"].isna()

print("Shape : ", usage.shape)
print(f"Missing active users : {usage['missing'].sum()} of {len(usage)} ({usage['missing'].mean():.1%})")

Shape :  (9540, 6)
Missing active users : 493 of 9540 (5.2%)


#### B. Where are the Gaps?

In [9]:
by_month = usage.groupby("month")["missing"].agg(missing="sum", share="mean")
print(by_month.round(3))

         missing  share
month                  
2025-10       32  0.040
2025-11       25  0.031
2025-12       24  0.030
2026-01       23  0.029
2026-02       27  0.034
2026-03      204  0.257
2026-04       21  0.026
2026-05       24  0.030
2026-06       27  0.034
2026-07       31  0.039
2026-08       27  0.034
2026-09       28  0.035


In [10]:
march = usage[usage["month"] == "2026-03"].merge(
    df[["subscription_id", "region"]], on="subscription_id", how="left"
)

by_region = march.groupby("region")["missing"].agg(missing="sum", rows="count", share="mean")
print(by_region.round(3))

          missing  rows  share
region                        
APJC          192   192  1.000
Americas        8   379  0.021
EMEA            4   224  0.018


#### C. Compare Five Ways of Filling the Gaps

`monthly_usage.csv` holds the true values, so each method is scored by its mean absolute error on the rows that were missing.

In [11]:
truth = pd.read_csv(DATA / "monthly_usage.csv")
truth = truth[["subscription_id", "month", "active_users"]].rename(columns={"active_users": "true_users"})

usage = usage.merge(truth, on=["subscription_id", "month"], how="left", validate="one_to_one")

gap = usage["missing"]
own = usage.groupby("subscription_id")["active_users"]

fills = {
    "overall mean": usage["active_users"].fillna(usage["active_users"].mean()),
    "own mean": usage["active_users"].fillna(own.transform("mean")),
    "forward fill": own.ffill(),
    "forward then backward fill": own.ffill().groupby(usage["subscription_id"]).bfill(),
    "linear interpolation": own.transform(lambda s: s.interpolate(limit_direction="both")),
}

scores = pd.DataFrame({
    method: {
        "mae": (filled[gap] - usage.loc[gap, "true_users"]).abs().mean(),
        "still_missing": filled[gap].isna().sum(),
    }
    for method, filled in fills.items()
}).T

print(scores.round(2).sort_values("mae"))

                               mae  still_missing
linear interpolation         31.54            0.0
forward then backward fill   44.43            0.0
forward fill                 45.09           34.0
own mean                     47.35            0.0
overall mean                918.23            0.0


Forward fill alone cannot fill a gap in a subscription's first month, so its error is measured on fewer rows. Linear interpolation has the lowest error; forward then backward fill is the method applied below, and it beats both means.

#### D. Apply Forward then Backward Fill

In [12]:
usage["active_users"] = usage.groupby("subscription_id")["active_users"].ffill()
usage["active_users"] = usage.groupby("subscription_id")["active_users"].bfill()
print("Missing active users after fill : ", usage["active_users"].isna().sum())

usage["utilization"] = usage["active_users"] / usage["seats_purchased"]
usage_util = usage.groupby("subscription_id")["utilization"].mean().round(3)

df["usage_util"] = df["subscription_id"].map(usage_util)

print("Subscriptions without usage_util : ", df["usage_util"].isna().sum())
print(df["usage_util"].describe().round(3))

log_step("3. usage_util added")

Missing active users after fill :  0
Subscriptions without usage_util :  0
count    795.000
mean       0.653
std        0.121
min        0.300
25%        0.572
50%        0.655
75%        0.738
max        0.983
Name: usage_util, dtype: float64
3. usage_util added               : 795 rows (-0)


### 4. Fix Invalid Values with Business Rules

`ACV = seats * list price * tier multiplier * (1 - discount%)`

When one of seats, discount and ACV is broken and the other two are fine, the broken one can be re-derived.

In [13]:
df["list_price_per_seat"] = (
    df["product"].map(rules["list_price_per_seat_per_year"])
    * df["tier"].map(rules["tier_multiplier"])
).round(2)

expected = df["seats_purchased"] * df["list_price_per_seat"] * (1 - df["discount_pct"] / 100)
off = (df["annual_contract_value"] / expected - 1).abs() > 0.01

print("Rows where ACV is more than 1% off the formula : ", off.sum(), "of", len(df))

Rows where ACV is more than 1% off the formula :  9 of 795


In [14]:
bad_seats = (df["seats_purchased"] < 0) | (df["seats_purchased"] >= 50000)
bad_discount = (df["discount_pct"] < 0) | (df["discount_pct"] > 40)
bad_acv = df["annual_contract_value"] <= 0

broken = pd.concat({"seats": bad_seats, "discount": bad_discount, "acv": bad_acv}, axis=1)
print(broken.sum().to_dict())
assert broken.sum(axis=1).max() <= 1, "a row has more than one broken value, it cannot be re-derived"

pricing = ["subscription_id", "seats_purchased", "discount_pct", "annual_contract_value"]
before = df.loc[broken.any(axis=1), pricing].copy()

{'seats': 3, 'discount': 4, 'acv': 2}


In [15]:
list_price = df["list_price_per_seat"]

# bad seats -> re-derive from ACV and discount
seats = df["annual_contract_value"] / (list_price * (1 - df["discount_pct"] / 100))
df.loc[bad_seats, "seats_purchased"] = seats[bad_seats].round().astype(int)

# bad discount -> re-derive from ACV and seats
discount = (1 - df["annual_contract_value"] / (df["seats_purchased"] * list_price)) * 100
df.loc[bad_discount, "discount_pct"] = discount[bad_discount].round(1)

# zero ACV -> re-derive from seats and discount
acv = df["seats_purchased"] * list_price * (1 - df["discount_pct"] / 100)
df.loc[bad_acv, "annual_contract_value"] = acv[bad_acv].round(2)

print(before.merge(df[pricing], on="subscription_id", suffixes=("_before", "_after")).T)

                                      0          1          2          3          4          5          6          7          8
subscription_id               SUB-50626  SUB-50987  SUB-50188  SUB-51189  SUB-51037  SUB-50910  SUB-50348  SUB-50281  SUB-50678
seats_purchased_before            99999       1130         25        125         50         20      99999      99999        140
discount_pct_before                11.9       -5.0        8.3       98.0       -5.0       92.0        0.0        8.1       11.4
annual_contract_value_before    70622.3   162310.0        0.0   20286.48   22570.96    7628.25    12600.0  109379.97        0.0
seats_purchased_after               165       1130         25        125         50         20         70        245        140
discount_pct_after                 11.9       20.2        8.3        9.8        7.1        9.2        0.0        8.1       11.4
annual_contract_value_after     70622.3   162310.0    5570.78   20286.48   22570.96    7628.25    12600.

In [16]:
as_percent = df["avg_seat_utilization"] > 5
df.loc[as_percent, "avg_seat_utilization"] = (df.loc[as_percent, "avg_seat_utilization"] / 100).round(3)

# still above 1 is impossible, so fall back to the telemetry
impossible = df["avg_seat_utilization"] > 1
df.loc[impossible, "avg_seat_utilization"] = df.loc[impossible, "usage_util"]

bad_nps = (df["nps_score"] < -100) | (df["nps_score"] > 100)
df.loc[bad_nps, "nps_score"] = np.nan

print("Utilization entered as a percentage : ", as_percent.sum())
print("Utilization replaced by usage_util  : ", impossible.sum())
print("NPS set to missing                  : ", bad_nps.sum())

Utilization entered as a percentage :  6
Utilization replaced by usage_util  :  3
NPS set to missing                  :  2


In [17]:
rule_checks = {
    "seats < 0 or >= 50000": (df["seats_purchased"] < 0) | (df["seats_purchased"] >= 50000),
    "discount outside 0-40%": (df["discount_pct"] < 0) | (df["discount_pct"] > 40),
    "ACV <= 0": df["annual_contract_value"] <= 0,
    "utilization outside 0-1": (df["avg_seat_utilization"] < 0) | (df["avg_seat_utilization"] > 1),
    "NPS outside -100 to 100": (df["nps_score"] < -100) | (df["nps_score"] > 100),
}

for rule, still_broken in rule_checks.items():
    print(f"{rule:<24}: {still_broken.sum()} rows")

log_step("4. invalid values fixed")

seats < 0 or >= 50000   : 0 rows
discount outside 0-40%  : 0 rows
ACV <= 0                : 0 rows
utilization outside 0-1 : 0 rows
NPS outside -100 to 100 : 0 rows
4. invalid values fixed           : 795 rows (-0)


### 5. Missing Values

In [18]:
df["churned"] = (df["renewal_status"] == "Churned").astype(int)

churn_by_nps = df.groupby(df["nps_score"].isna().map({True: "NPS missing", False: "NPS present"}))["churned"]
print(churn_by_nps.agg(rows="count", churn_rate="mean").round(3))

             rows  churn_rate
nps_score                    
NPS missing   137       0.482
NPS present   658       0.173


Customers with no NPS churn far more often, so the gap itself is a signal. Keep it as an indicator before filling.

In [19]:
df["nps_missing"] = df["nps_score"].isna().astype(int)
df["nps_score"] = df["nps_score"].fillna(df["nps_score"].median())

df["avg_feature_adoption"] = df["avg_feature_adoption"].fillna(df["avg_feature_adoption"].median())
df["industry"] = df["industry"].fillna("Unknown")
df["employee_count"] = df["employee_count"].fillna(
    df.groupby("segment")["employee_count"].transform("median")
)
df["sales_channel"] = df["sales_channel"].fillna(df["sales_channel"].mode()[0])

In [20]:
# these gaps are structural, not errors: there is nothing to report without a CSM
print(pd.crosstab(df["has_csm"], df["csm_health"].isna(), colnames=["csm_health missing"]))

df["csm_health"] = df["csm_health"].fillna("No CSM")
df["last_csm_note"] = df["last_csm_note"].fillna("No Note")

csm_health missing  False  True 
has_csm                         
False                   0    401
True                  394      0


In [21]:
# renewal_acv and churn_reason are outcomes of the renewal, so they stay missing
print(pd.crosstab(df["renewal_status"], df["renewal_acv"].isna(), colnames=["renewal_acv missing"]))
print()
print(pd.crosstab(df["renewal_status"], df["churn_reason"].isna(), colnames=["churn_reason missing"]))

miss = df.isna().sum()
print()
print("Columns still holding missing values : ", miss[miss > 0].to_dict())

log_step("5. missing values filled")

renewal_acv missing  False  True 
renewal_status                   
Churned                  0    180
Renewed                615      0

churn_reason missing  False  True 
renewal_status                    
Churned                 180      0
Renewed                   0    615

Columns still holding missing values :  {'renewal_acv': 180, 'churn_reason': 615}
5. missing values filled          : 795 rows (-0)


### 6. Outliers with the IQR Rule

In [22]:
def iqr_bounds(values):
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


outlier_columns = ["annual_contract_value", "seats_purchased", "support_cases_12m",
                   "avg_seat_utilization", "nps_score"]

for column in outlier_columns:
    low, high = iqr_bounds(df[column])
    outliers = (df[column] < low) | (df[column] > high)
    print(f"{column:<22}: {outliers.sum():>3} outliers ({outliers.mean():.1%})"
          f" | bounds {low:,.2f} to {high:,.2f} | data {df[column].min():,.2f} to {df[column].max():,.2f}")

annual_contract_value : 102 outliers (12.8%) | bounds -432,573.04 to 774,588.69 | data 2,436.42 to 4,169,814.56
seats_purchased       : 100 outliers (12.6%) | bounds -1,673.75 to 3,016.25 | data 15.00 to 11,790.00
support_cases_12m     :  29 outliers (3.6%) | bounds -3.50 to 16.50 | data 0.00 to 32.00
avg_seat_utilization  :   4 outliers (0.5%) | bounds 0.33 to 0.98 | data 0.30 to 0.98
nps_score             :  47 outliers (5.9%) | bounds 5.00 to 69.00 | data -16.00 to 84.00


In [23]:
log_acv = np.log10(df["annual_contract_value"])
low, high = iqr_bounds(log_acv)
outliers = (log_acv < low) | (log_acv > high)

print(f"log10(ACV) : {outliers.sum()} outliers | bounds {low:.2f} to {high:.2f}"
      f" | data {log_acv.min():.2f} to {log_acv.max():.2f}")

log10(ACV) : 0 outliers | bounds 2.50 to 7.31 | data 3.39 to 6.62


**Errors or genuine?** The errors were already repaired in step 4, so everything the IQR rule still flags is genuine and stays in the data.

| Column | Verdict |
| --- | --- |
| ACV | Genuine. Large enterprise contracts; the outliers disappear on the log scale. |
| Seats | Genuine. Same right skew as ACV, the big accounts buy thousands of seats. |
| Support cases | Genuine. A few accounts raise many cases, which is a churn signal. |
| Utilization | Genuine. All values sit inside the valid 0 to 1 range. |
| NPS | Genuine. All values sit inside the valid -100 to 100 range. The median fill in step 5 narrowed the IQR, so more scores are flagged here than among the reported scores alone. |

In [24]:
log_step("6. outliers reviewed")

6. outliers reviewed              : 795 rows (-0)


### Save the Clean File

In [25]:
OUT.mkdir(exist_ok=True)
df.to_csv(OUT / "renewals_clean.csv", index=False)

print("Saved : ", OUT / "renewals_clean.csv", df.shape)
pd.DataFrame(log)

Saved :  ..\output\renewals_clean.csv (795, 34)


,step,rows,dropped
0,raw extract,813,0
1,1. standardized,813,0
2,2a. exact duplicates dropped,801,12
3,2b. duplicate ids dropped,795,6
4,3. usage_util added,795,0
5,4. invalid values fixed,795,0
6,5. missing values filled,795,0
7,6. outliers reviewed,795,0
